# Robust + Explainable AI for Diabetic Retinopathy — All Phases

Single-notebook version. Assumes you've already mounted Drive and extracted `Thesis.zip` so that `/content/data/aptos/` and `/content/data/eyeq/` are populated.

**Sections (run top-to-bottom, each section is independent within a session):**

0. Shared setup — config, imports, dataset classes, model factory, XAI helpers, metric helpers
1. **Phase 1** — Data engineering & synthetic degradation
2. **Phase 2 (RQ1)** — Train ResNet-50 / EfficientNet-B3 / ViT-Base, stress-test
3. **Phase 3 (RQ2)** — Grad-CAM / SHAP / Attention + faithfulness / stability / localization
4. **Phase 4 (RQ3)** — CLAHE + GenAI restoration, accuracy & XAI recovery
5. **Phase 5 (RQ4)** — Quality-aware ensemble pipeline + clinical trust score

Every phase writes to its own `results/phaseN_*/` folder on Drive — they stay separable for the professor pitch.

## 0. Shared setup

Run **all** cells in this section once per Colab session. Subsequent phase sections depend on the names defined here.

In [ ]:
%%capture
!pip install -q timm captum shap scikit-image opencv-python-headless tabulate diffusers transformers accelerate

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

### 0.1 Master config & paths

In [ ]:
from pathlib import Path
import torch

# Drive (persistent)
DRIVE_ROOT      = Path('/content/drive/MyDrive/Thesis')
RESULTS_ROOT    = DRIVE_ROOT / 'results'
CHECKPOINTS_DIR = DRIVE_ROOT / 'checkpoints'

# Local Colab disk (working) -- already populated
LOCAL_ROOT   = Path('/content/data')
APTOS_DIR    = LOCAL_ROOT / 'aptos'
EYEQ_DIR     = LOCAL_ROOT / 'eyeq'
DEGRADED_DIR = LOCAL_ROOT / 'degraded'
ENHANCED_DIR = LOCAL_ROOT / 'enhanced'

PHASE_DIRS = {
    'phase1_data_engineering':   RESULTS_ROOT / 'phase1_data_engineering',
    'phase2_model_benchmarking': RESULTS_ROOT / 'phase2_model_benchmarking',
    'phase3_xai_benchmark':      RESULTS_ROOT / 'phase3_xai_benchmark',
    'phase4_genai_enhancement':  RESULTS_ROOT / 'phase4_genai_enhancement',
    'phase5_quality_ensemble':   RESULTS_ROOT / 'phase5_quality_ensemble',
}
for p in [DEGRADED_DIR, ENHANCED_DIR, RESULTS_ROOT, CHECKPOINTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)
for d in PHASE_DIRS.values():
    for sub in ('metrics', 'plots', 'samples', 'logs'):
        (d / sub).mkdir(parents=True, exist_ok=True)

# Auto-detect APTOS train CSV + image dir (case-insensitive)
APTOS_CSV    = next(p for p in APTOS_DIR.rglob('*.csv') if 'train' in p.name.lower())
APTOS_IMAGES = next(p for p in APTOS_DIR.rglob('*train_images*') if p.is_dir())
EYEQ_CSV     = next(EYEQ_DIR.rglob('*.csv'), None)

# Common constants
NUM_CLASSES = 5
CLASS_NAMES = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
IMAGE_SIZE  = 224
SEED        = 42
VAL_SPLIT, TEST_SPLIT = 0.15, 0.15

MODEL_NAMES = ('resnet50', 'efficientnet_b3', 'vit_base')
TIMM_NAMES  = {'resnet50': 'resnet50',
               'efficientnet_b3': 'efficientnet_b3',
               'vit_base': 'vit_base_patch16_224'}
TRAIN_CFG = dict(batch_size=32, num_workers=2, epochs=15,
                 lr=3e-4, weight_decay=1e-4, label_smoothing=0.05,
                 mixed_precision=True)

DEGRADATION_TYPES  = ('blur', 'exposure', 'noise')
DEGRADATION_LEVELS = ('low', 'mid', 'high')
DEGRADATION_PARAMS = {
    'blur':     {'low': 2.0,  'mid': 5.0,  'high': 9.0},
    'exposure': {'low': 0.7,  'mid': 0.4,  'high': 0.2},
    'noise':    {'low': 0.02, 'mid': 0.06, 'high': 0.12},
}
EYEQ_QUALITY_LABELS = {0: 'good', 1: 'usable', 2: 'reject'}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('APTOS csv   :', APTOS_CSV)
print('APTOS images:', APTOS_IMAGES)
print('EyeQ csv    :', EYEQ_CSV)
print('Device      :', DEVICE)

### 0.2 Imports + transforms

In [ ]:
import json, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.cuda.amp import GradScaler, autocast
from torchvision import transforms as T
import timm
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from scipy.stats import spearmanr
from skimage.segmentation import slic

TFM_TRAIN = T.Compose([T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                        T.RandomHorizontalFlip(),
                        T.RandomRotation(15),
                        T.ColorJitter(0.1, 0.1, 0.1),
                        T.ToTensor(),
                        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
TFM_EVAL  = T.Compose([T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                        T.ToTensor(),
                        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

def load_tensor(path: Path) -> torch.Tensor:
    return TFM_EVAL(Image.open(path).convert('RGB')).unsqueeze(0).to(DEVICE)

### 0.3 Dataset classes

In [ ]:
class APTOSDataset(Dataset):
    def __init__(self, csv_path, images_dir, transform):
        self.df = pd.read_csv(csv_path)
        self.images_dir = Path(images_dir); self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(self.images_dir / f"{row['id_code']}.png").convert('RGB')
        return self.transform(img), int(row['diagnosis']), row['id_code']

class FolderDataset(Dataset):
    """Reads `<root>/manifest.csv` (cols: id_code, diagnosis, rel_path)."""
    def __init__(self, root, transform=TFM_EVAL):
        self.root = Path(root); self.df = pd.read_csv(self.root / 'manifest.csv')
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(self.root / row['rel_path']).convert('RGB')
        return self.transform(img), int(row['diagnosis']), row['id_code']

### 0.4 Model factory + train/evaluate helpers

In [ ]:
def build_model(name, pretrained=True):
    m = timm.create_model(TIMM_NAMES[name], pretrained=pretrained, num_classes=NUM_CLASSES)
    m._dr_arch = name
    return m

def load_classifier(name):
    m = build_model(name, pretrained=False).to(DEVICE)
    state = torch.load(CHECKPOINTS_DIR / f'{name}_best.pt', map_location=DEVICE)
    m.load_state_dict(state['state_dict']); m.eval()
    return m

def get_xai_target_layer(model):
    a = model._dr_arch
    if a == 'resnet50':        return model.layer4[-1]
    if a == 'efficientnet_b3': return model.blocks[-1]
    if a == 'vit_base':        return model.blocks[-1].norm1
    raise ValueError(a)

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); ys, ps, probs = [], [], []
    for x, y, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        prob = torch.softmax(model(x), 1).cpu().numpy()
        probs.append(prob); ps.append(prob.argmax(1)); ys.append(y.numpy())
    y = np.concatenate(ys); p = np.concatenate(ps); pr = np.concatenate(probs)
    out = {'accuracy': accuracy_score(y, p), 'f1_macro': f1_score(y, p, average='macro')}
    try: out['auc_macro_ovr'] = roc_auc_score(y, pr, average='macro', multi_class='ovr')
    except ValueError: out['auc_macro_ovr'] = float('nan')
    return out

def train_one(model, train_loader, val_loader, ckpt_path, history_csv,
              epochs=TRAIN_CFG['epochs'], lr=TRAIN_CFG['lr'],
              wd=TRAIN_CFG['weight_decay'], ls=TRAIN_CFG['label_smoothing'],
              amp=TRAIN_CFG['mixed_precision']):
    model = model.to(DEVICE)
    optim  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
    crit   = nn.CrossEntropyLoss(label_smoothing=ls)
    scaler = GradScaler(enabled=amp)
    best_f1, history = -1.0, []
    for ep in range(epochs):
        model.train(); running, seen = 0.0, 0
        pbar = tqdm(train_loader, desc=f'epoch {ep+1}/{epochs}', leave=False)
        for x, y, _ in pbar:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE)
            optim.zero_grad(set_to_none=True)
            with autocast(enabled=amp):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(optim); scaler.update()
            running += loss.item() * x.size(0); seen += x.size(0)
            pbar.set_postfix(loss=running/seen)
        sched.step()
        m = evaluate(model, val_loader)
        history.append({'epoch': ep+1, 'train_loss': running/seen, **m})
        print(f'  val: acc={m["accuracy"]:.4f}  f1={m["f1_macro"]:.4f}  auc={m["auc_macro_ovr"]:.4f}')
        if m['f1_macro'] > best_f1:
            best_f1 = m['f1_macro']
            torch.save({'state_dict': model.state_dict(), 'arch': model._dr_arch, 'val': m}, ckpt_path)
    pd.DataFrame(history).to_csv(history_csv, index=False)
    return pd.DataFrame(history)

### 0.5 XAI heatmap functions (Grad-CAM, Attention rollout, KernelSHAP)

In [ ]:
def gradcam_heatmap(model, x, target_class=None):
    layer = get_xai_target_layer(model)
    acts, grads = {}, {}
    h1 = layer.register_forward_hook(lambda _, __, o: acts.update(v=o.detach()))
    h2 = layer.register_full_backward_hook(lambda _, gi, go: grads.update(v=go[0].detach()))
    try:
        model.zero_grad(set_to_none=True)
        x = x.clone().requires_grad_(True)
        logits = model(x)
        if target_class is None:
            target_class = int(logits.argmax(1).item())
        logits[0, target_class].backward()
        a, g = acts['v'][0], grads['v'][0]
        cam = torch.relu((g.mean(dim=(1, 2))[:, None, None] * a).sum(0))
        cam -= cam.min()
        if cam.max() > 0: cam /= cam.max()
        cam = F.interpolate(cam[None, None], size=x.shape[-2:], mode='bilinear', align_corners=False)
        return cam.squeeze().detach().cpu().numpy()
    finally:
        h1.remove(); h2.remove()

@torch.no_grad()
def attention_rollout(model, x, target_class=None):
    attns, handles = [], []
    for blk in model.blocks:
        attn_module = blk.attn
        def make_hook(store, mod):
            def _h(module, inp, out):
                B, N, C = inp[0].shape
                qkv = mod.qkv(inp[0]).reshape(B, N, 3, mod.num_heads, C // mod.num_heads).permute(2, 0, 3, 1, 4)
                q, k, _ = qkv[0], qkv[1], qkv[2]
                a = (q @ k.transpose(-2, -1)) * mod.scale
                store.append(a.softmax(-1).detach())
            return _h
        handles.append(attn_module.register_forward_hook(make_hook(attns, attn_module)))
    try: _ = model(x)
    finally:
        for h in handles: h.remove()
    res = None
    for a in attns:
        a = a.mean(1) + torch.eye(a.size(-1), device=a.device)
        a = a / a.sum(-1, keepdim=True)
        res = a if res is None else a @ res
    cls = res[0, 0, 1:]; side = int(cls.numel() ** 0.5)
    h = F.interpolate(cls.reshape(side, side)[None, None],
                       size=x.shape[-2:], mode='bilinear', align_corners=False).squeeze().cpu().numpy()
    if h.max() > h.min(): h = (h - h.min()) / (h.max() - h.min())
    return h

def shap_heatmap(model, x, target_class=None, n_samples=60, n_segments=40):
    import shap
    img = x.squeeze(0).permute(1, 2, 0).cpu().numpy()
    seg = slic(img, n_segments=n_segments, compactness=10, start_label=0)
    def f(masks):
        out = np.zeros((masks.shape[0], NUM_CLASSES), dtype=np.float32)
        for i, m in enumerate(masks):
            keep = np.isin(seg, np.where(m == 1)[0])
            masked = img.copy(); masked[~keep] = img.mean(axis=(0, 1))
            t = torch.tensor(masked).permute(2, 0, 1).unsqueeze(0).float().to(DEVICE)
            with torch.no_grad():
                out[i] = torch.softmax(model(t), 1).cpu().numpy()[0]
        return out
    n_super = int(seg.max() + 1)
    explainer = shap.KernelExplainer(f, np.zeros((1, n_super)))
    sv = explainer.shap_values(np.ones((1, n_super)), nsamples=n_samples)
    if target_class is None:
        with torch.no_grad():
            target_class = int(model(x).argmax(1).item())
    sv = sv[target_class][0]
    heat = np.zeros(seg.shape, dtype=np.float32)
    for sid, val in enumerate(sv):
        heat[seg == sid] = val
    if heat.max() > heat.min():
        heat = (heat - heat.min()) / (heat.max() - heat.min())
    return heat

### 0.6 XAI metric helpers (faithfulness, stability, localization)

In [ ]:
@torch.no_grad()
def _prob_for(model, x, cls): return torch.softmax(model(x), 1)[0, cls].item()

def deletion_auc(model, x, heatmap, target_class, steps=30):
    H, W = heatmap.shape
    order = np.argsort(-heatmap.flatten())
    chunk = max(1, len(order) // steps)
    mean_pix = x.mean(dim=(2, 3), keepdim=True)
    probs = []
    for s in range(steps + 1):
        idx = order[: s * chunk]
        rows, cols = np.unravel_index(idx, (H, W))
        x_mod = x.clone()
        x_mod[0, :, rows, cols] = mean_pix.expand_as(x)[0, :, rows, cols]
        probs.append(_prob_for(model, x_mod, target_class))
    return float(np.trapz(probs, dx=1/steps))

def insertion_auc(model, x, heatmap, target_class, steps=30):
    H, W = heatmap.shape
    order = np.argsort(-heatmap.flatten())
    chunk = max(1, len(order) // steps)
    base = F.avg_pool2d(x, 15, stride=1, padding=7)
    probs = []
    for s in range(steps + 1):
        idx = order[: s * chunk]
        rows, cols = np.unravel_index(idx, (H, W))
        x_mod = base.clone(); x_mod[0, :, rows, cols] = x[0, :, rows, cols]
        probs.append(_prob_for(model, x_mod, target_class))
    return float(np.trapz(probs, dx=1/steps))

def stability_spearman(h1, h2):
    rho, _ = spearmanr(h1.flatten(), h2.flatten())
    return float(rho) if rho == rho else 0.0

def localization_iou(heatmap, mask, percentile=80):
    if mask is None: return float('nan')
    thr  = np.percentile(heatmap, percentile)
    pred = heatmap >= thr
    inter = np.logical_and(pred, mask).sum()
    union = np.logical_or(pred, mask).sum()
    return float(inter/union) if union > 0 else float('nan')

---
# Phase 1 — Data Engineering & Synthetic Degradation
Outputs: `Drive/Thesis/results/phase1_data_engineering/`

In [ ]:
P1 = PHASE_DIRS['phase1_data_engineering']

# 1.1 Quality filter (EyeQ -> pristine subset; falls back to full APTOS)
def filter_pristine(aptos_csv, eyeq_csv, out_csv,
                    keep_quality=('good',),
                    eyeq_id_col='image', eyeq_quality_col='quality',
                    aptos_id_col='id_code'):
    aptos = pd.read_csv(aptos_csv)
    if eyeq_csv is None or not Path(eyeq_csv).exists():
        print('[fallback] No EyeQ CSV — keeping all APTOS rows.')
        aptos.to_csv(out_csv, index=False); return aptos
    eyeq = pd.read_csv(eyeq_csv)
    if pd.api.types.is_integer_dtype(eyeq[eyeq_quality_col]):
        eyeq['__q'] = eyeq[eyeq_quality_col].map(EYEQ_QUALITY_LABELS)
    else:
        eyeq['__q'] = eyeq[eyeq_quality_col].astype(str).str.lower()
    keep_ids = set(eyeq.loc[eyeq['__q'].isin(keep_quality), eyeq_id_col].astype(str))
    pristine = aptos[aptos[aptos_id_col].astype(str).isin(keep_ids)].reset_index(drop=True)
    if len(pristine) == 0:
        print('[warn] EyeQ filter removed all rows — falling back to full APTOS.')
        aptos.to_csv(out_csv, index=False); return aptos
    pristine.to_csv(out_csv, index=False); return pristine

PRISTINE_CSV = P1 / 'metrics' / 'pristine_split.csv'
pristine_df = filter_pristine(APTOS_CSV, EYEQ_CSV, PRISTINE_CSV)
print(f'kept {len(pristine_df)} / {pd.read_csv(APTOS_CSV).shape[0]} images')
pristine_df.head()

In [ ]:
# 1.2 Class-balance plot
counts = pristine_df['diagnosis'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar([CLASS_NAMES[i] for i in counts.index], counts.values, color='steelblue')
for b, v in zip(bars, counts.values):
    ax.text(b.get_x() + b.get_width()/2, v, str(v), ha='center', va='bottom', fontsize=9)
ax.set_title('Pristine class distribution'); ax.set_ylabel('count')
plt.xticks(rotation=15); plt.tight_layout()
out = P1 / 'plots' / 'class_distribution.png'
plt.savefig(out, dpi=150); plt.show()
print('Saved:', out)

In [ ]:
# 1.3 Synthetic degradation primitives
def _np(img):  return np.array(img.convert('RGB'))
def _pil(arr): return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

def gaussian_blur(img, sigma):
    k = max(3, int(2 * round(3 * sigma) + 1))
    return _pil(cv2.GaussianBlur(_np(img), (k, k), sigmaX=sigma, sigmaY=sigma))

def exposure_shift(img, gain): return _pil(_np(img).astype(np.float32) * gain)
def gaussian_noise(img, std_frac):
    arr = _np(img).astype(np.float32)
    return _pil(arr + np.random.normal(0, std_frac * 255, arr.shape))

DEGRADERS = {'blur': gaussian_blur, 'exposure': exposure_shift, 'noise': gaussian_noise}
def apply_degradation(img, kind, level):
    return DEGRADERS[kind](img, DEGRADATION_PARAMS[kind][level])

In [ ]:
# 1.4 Build all 9 (type, level) splits + manifests
def build_split(kind, level):
    out_dir = DEGRADED_DIR / kind / level
    out_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for _, row in tqdm(pristine_df.iterrows(), total=len(pristine_df),
                       desc=f'{kind}-{level}', leave=False):
        src = APTOS_IMAGES / f"{row['id_code']}.png"
        if not src.exists(): continue
        out_img = apply_degradation(Image.open(src).convert('RGB'), kind, level)
        out_img.save(out_dir / f"{row['id_code']}.png")
        rows.append({'id_code': row['id_code'], 'diagnosis': int(row['diagnosis']),
                     'degradation': kind, 'level': level,
                     'rel_path': f"{row['id_code']}.png"})
    df = pd.DataFrame(rows)
    df.to_csv(out_dir / 'manifest.csv', index=False)
    df.to_csv(P1 / 'metrics' / f'manifest_{kind}_{level}.csv', index=False)
    return df

for k in DEGRADATION_TYPES:
    for l in DEGRADATION_LEVELS:
        m = build_split(k, l)
        print(f'  {k}/{l}: {len(m)} images')

In [ ]:
# 1.5 Sample grids — clean / low / mid / high per type
rng = np.random.default_rng(SEED)
sample_ids = rng.choice(pristine_df['id_code'].values, size=4, replace=False)
for kind in DEGRADATION_TYPES:
    fig, axes = plt.subplots(4, 4, figsize=(11, 11))
    for r, sid in enumerate(sample_ids):
        clean = Image.open(APTOS_IMAGES / f'{sid}.png').convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
        axes[r, 0].imshow(clean); axes[r, 0].set_title('clean' if r == 0 else '')
        for c, lvl in enumerate(DEGRADATION_LEVELS, start=1):
            axes[r, c].imshow(apply_degradation(clean, kind, lvl))
            if r == 0: axes[r, c].set_title(lvl)
        for ax in axes[r]: ax.axis('off')
    fig.suptitle(f'Degradation: {kind}', fontsize=14); plt.tight_layout()
    out = P1 / 'samples' / f'grid_{kind}.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
    print('Saved:', out)

# 1.6 Phase summary
summary = pd.DataFrame([{'degradation': k, 'level': l,
                          'n_images': len(list((DEGRADED_DIR/k/l).glob('*.png')))}
                         for k in DEGRADATION_TYPES for l in DEGRADATION_LEVELS])
summary.to_csv(P1 / 'metrics' / 'phase1_summary.csv', index=False)
summary

---
# Phase 2 — Model Benchmarking (RQ1)
Outputs: `Drive/Thesis/results/phase2_model_benchmarking/` + `Drive/Thesis/checkpoints/<model>_best.pt`

In [ ]:
P2 = PHASE_DIRS['phase2_model_benchmarking']
PRISTINE_CSV = P1 / 'metrics' / 'pristine_split.csv'   # carry forward from Phase 1

# 2.1 Train/val/test split (deterministic)
full_pristine_train = APTOSDataset(PRISTINE_CSV, APTOS_IMAGES, TFM_TRAIN)
full_pristine_eval  = APTOSDataset(PRISTINE_CSV, APTOS_IMAGES, TFM_EVAL)
n = len(full_pristine_train)
n_test = int(n * TEST_SPLIT); n_val = int(n * VAL_SPLIT); n_train = n - n_val - n_test
g = torch.Generator().manual_seed(SEED)
train_ds, val_ds, test_ds = random_split(full_pristine_train, [n_train, n_val, n_test], generator=g)
for ds in (val_ds, test_ds):
    ds.dataset = full_pristine_eval                 # eval transforms (no aug)

common = dict(batch_size=TRAIN_CFG['batch_size'], num_workers=TRAIN_CFG['num_workers'], pin_memory=True)
train_loader = DataLoader(train_ds, shuffle=True,  **common)
val_loader   = DataLoader(val_ds,   shuffle=False, **common)
test_loader  = DataLoader(test_ds,  shuffle=False, **common)

test_ids = [test_ds.dataset.df.iloc[i]['id_code'] for i in test_ds.indices]
pd.DataFrame({'id_code': test_ids}).to_csv(P2 / 'metrics' / 'test_ids.csv', index=False)
print(f'train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}')

In [ ]:
# 2.2 Train all three models
histories = {}
for name in MODEL_NAMES:
    print(f'\n========== Training {name} ==========')
    model = build_model(name, pretrained=True)
    histories[name] = train_one(model, train_loader, val_loader,
                                 ckpt_path=CHECKPOINTS_DIR / f'{name}_best.pt',
                                 history_csv=P2 / 'metrics' / f'training_history_{name}.csv')
    del model; torch.cuda.empty_cache()

In [ ]:
# 2.3 Training curves
for name, h in histories.items():
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(h['epoch'], h['train_loss']); ax[0].set_title(f'{name} — train loss')
    ax[0].set_xlabel('epoch'); ax[0].set_ylabel('loss')
    ax[1].plot(h['epoch'], h['accuracy'], label='val acc')
    ax[1].plot(h['epoch'], h['f1_macro'], label='val F1')
    ax[1].set_title(f'{name} — val metrics'); ax[1].legend(); ax[1].set_xlabel('epoch')
    plt.tight_layout()
    out = P2 / 'plots' / f'training_curves_{name}.png'
    plt.savefig(out, dpi=150); plt.show(); print('Saved:', out)

In [ ]:
# 2.4 Stress test — every model on clean + 9 degraded variants
test_id_set = set(map(str, test_ids))

def deg_loader(kind, level):
    ds = FolderDataset(DEGRADED_DIR / kind / level)
    ds.df = ds.df[ds.df['id_code'].astype(str).isin(test_id_set)].reset_index(drop=True)
    return DataLoader(ds, shuffle=False, **common)

rows = []
for name in MODEL_NAMES:
    print(f'\n--- {name} ---')
    model = load_classifier(name)
    m = evaluate(model, test_loader)
    rows.append({'model': name, 'degradation': 'clean', 'level': 'none', **m})
    print(f'  clean: acc={m["accuracy"]:.4f}  f1={m["f1_macro"]:.4f}')
    for k in DEGRADATION_TYPES:
        for l in DEGRADATION_LEVELS:
            m = evaluate(model, deg_loader(k, l))
            rows.append({'model': name, 'degradation': k, 'level': l, **m})
            print(f'  {k}/{l}: acc={m["accuracy"]:.4f}  f1={m["f1_macro"]:.4f}')
    del model; torch.cuda.empty_cache()

stress_df = pd.DataFrame(rows)
stress_df.to_csv(P2 / 'metrics' / 'stress_test_results.csv', index=False)
stress_df

In [ ]:
# 2.5 Accuracy-vs-degradation plots
level_order = ['none', 'low', 'mid', 'high']
for kind in DEGRADATION_TYPES:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for name in MODEL_NAMES:
        sub = stress_df[((stress_df['degradation'] == kind) | (stress_df['degradation'] == 'clean'))
                        & (stress_df['model'] == name)].copy()
        sub['level'] = pd.Categorical(sub['level'], categories=level_order, ordered=True)
        sub = sub.sort_values('level')
        ax.plot(sub['level'], sub['accuracy'], marker='o', label=name)
    ax.set_title(f'Accuracy vs {kind} severity')
    ax.set_xlabel('level'); ax.set_ylabel('accuracy')
    ax.set_ylim(0, 1); ax.grid(alpha=0.3); ax.legend()
    plt.tight_layout()
    out = P2 / 'plots' / f'accuracy_vs_degradation_{kind}.png'
    plt.savefig(out, dpi=150); plt.show(); print('Saved:', out)

pivot = stress_df.pivot_table(index=['degradation', 'level'], columns='model', values='accuracy').round(3)
pivot.to_csv(P2 / 'metrics' / 'accuracy_pivot.csv'); pivot

---
# Phase 3 — Explainability Benchmark (RQ2)
Outputs: `Drive/Thesis/results/phase3_xai_benchmark/`

In [ ]:
P3 = PHASE_DIRS['phase3_xai_benchmark']

# 3.1 Pick the same N test images we'll explain across every condition
N_EXPLAIN = 20                      # bump for final run; SHAP is the bottleneck
test_ids_full = pd.read_csv(P2 / 'metrics' / 'test_ids.csv')['id_code'].tolist()
EXPLAIN_IDS = pd.Series(test_ids_full).sample(N_EXPLAIN, random_state=SEED).tolist()
labels_df = pd.read_csv(PRISTINE_CSV).set_index('id_code')['diagnosis']

models = {n: load_classifier(n) for n in MODEL_NAMES}
print('Loaded:', list(models))

# Optional ground-truth lesion masks at /content/data/lesion_masks/<id>.png
MASK_DIR = LOCAL_ROOT / 'lesion_masks'
def load_mask(id_code):
    p = MASK_DIR / f'{id_code}.png'
    if not p.exists(): return None
    return (np.array(Image.open(p).convert('L').resize((IMAGE_SIZE, IMAGE_SIZE))) > 127)

In [ ]:
# 3.2 Run XAI benchmark — for each (image, model, method, condition)
METHOD_REGISTRY = {
    'gradcam':   {'fn': gradcam_heatmap,    'models': ('resnet50', 'efficientnet_b3')},
    'shap':      {'fn': shap_heatmap,       'models': MODEL_NAMES},
    'attention': {'fn': attention_rollout,  'models': ('vit_base',)},
}

rows = []
for method_name, spec in METHOD_REGISTRY.items():
    fn = spec['fn']
    for model_name in spec['models']:
        model = models[model_name]
        print(f'\n[{method_name} | {model_name}]')
        for id_code in tqdm(EXPLAIN_IDS, desc=f'{method_name}/{model_name}'):
            label = int(labels_df[id_code])
            mask  = load_mask(id_code)
            x_clean = load_tensor(APTOS_IMAGES / f'{id_code}.png')
            try:    h_clean = fn(model, x_clean, target_class=label)
            except Exception as e: print('  skip clean:', e); continue
            rows.append({'id_code': id_code, 'model': model_name, 'method': method_name,
                         'degradation': 'clean', 'level': 'none',
                         'deletion_auc':  deletion_auc(model, x_clean, h_clean, label),
                         'insertion_auc': insertion_auc(model, x_clean, h_clean, label),
                         'stability':     1.0,
                         'iou':           localization_iou(h_clean, mask)})
            for k in DEGRADATION_TYPES:
                for l in DEGRADATION_LEVELS:
                    src = DEGRADED_DIR / k / l / f'{id_code}.png'
                    if not src.exists(): continue
                    x_deg = load_tensor(src)
                    try: h_deg = fn(model, x_deg, target_class=label)
                    except Exception: continue
                    rows.append({'id_code': id_code, 'model': model_name, 'method': method_name,
                                 'degradation': k, 'level': l,
                                 'deletion_auc':  deletion_auc(model, x_deg, h_deg, label),
                                 'insertion_auc': insertion_auc(model, x_deg, h_deg, label),
                                 'stability':     stability_spearman(h_clean, h_deg),
                                 'iou':           localization_iou(h_deg, mask)})

xai_df = pd.DataFrame(rows)
xai_df.to_csv(P3 / 'metrics' / 'xai_results.csv', index=False)
xai_df.head()

In [ ]:
# 3.3 Aggregated tables + plots
for metric in ('deletion_auc', 'insertion_auc', 'stability', 'iou'):
    summary = (xai_df.groupby(['model', 'method', 'degradation', 'level'])[metric]
                       .agg(['mean', 'std', 'count']).round(4).reset_index())
    summary.to_csv(P3 / 'metrics' / f'summary_{metric}.csv', index=False)

for metric in ('deletion_auc', 'insertion_auc', 'stability', 'iou'):
    for kind in DEGRADATION_TYPES:
        sub = xai_df[xai_df['degradation'].isin([kind, 'clean'])].copy()
        sub['level'] = pd.Categorical(sub['level'], categories=level_order, ordered=True)
        if sub[metric].dropna().empty: continue
        g = sub.groupby(['model', 'method', 'level'])[metric].mean().reset_index()
        fig, ax = plt.subplots(figsize=(7, 4.5))
        for (m, mt), grp in g.groupby(['model', 'method']):
            ax.plot(grp['level'], grp[metric], marker='o', label=f'{m}/{mt}')
        ax.set_title(f'{metric} vs {kind}')
        ax.set_xlabel('level'); ax.set_ylabel(metric); ax.legend(fontsize=8); ax.grid(alpha=0.3)
        plt.tight_layout()
        out = P3 / 'plots' / f'{metric}_vs_{kind}.png'
        plt.savefig(out, dpi=150); plt.show()

In [ ]:
# 3.4 Qualitative — clean vs blur-high heatmap per model
def overlay(img_pil, heat, alpha=0.45):
    arr = np.array(img_pil.resize((IMAGE_SIZE, IMAGE_SIZE))).astype(np.float32) / 255
    return (1 - alpha) * arr + alpha * plt.get_cmap('jet')(heat)[..., :3]

demo_id = EXPLAIN_IDS[0]; label = int(labels_df[demo_id])
img_clean = Image.open(APTOS_IMAGES / f'{demo_id}.png').convert('RGB')
x_clean = load_tensor(APTOS_IMAGES / f'{demo_id}.png')
img_deg = Image.open(DEGRADED_DIR / 'blur' / 'high' / f'{demo_id}.png').convert('RGB')
x_deg = load_tensor(DEGRADED_DIR / 'blur' / 'high' / f'{demo_id}.png')

fig, axes = plt.subplots(len(MODEL_NAMES), 4, figsize=(13, 3.2*len(MODEL_NAMES)))
for r, name in enumerate(MODEL_NAMES):
    method = 'attention' if name == 'vit_base' else 'gradcam'
    fn = METHOD_REGISTRY[method]['fn']
    h_c = fn(models[name], x_clean, target_class=label)
    h_d = fn(models[name], x_deg,   target_class=label)
    axes[r, 0].imshow(img_clean.resize((IMAGE_SIZE, IMAGE_SIZE))); axes[r, 0].set_title(f'{name} clean' if r == 0 else 'clean')
    axes[r, 1].imshow(overlay(img_clean, h_c)); axes[r, 1].set_title(f'{method} clean')
    axes[r, 2].imshow(img_deg.resize((IMAGE_SIZE, IMAGE_SIZE))); axes[r, 2].set_title('blur-high')
    axes[r, 3].imshow(overlay(img_deg, h_d)); axes[r, 3].set_title(f'{method} blur-high')
    for ax in axes[r]: ax.axis('off')
plt.tight_layout()
out = P3 / 'samples' / f'qualitative_{demo_id}.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print('Saved:', out)

---
# Phase 4 — GenAI Enhancement & XAI Recovery (RQ3)
Outputs: `Drive/Thesis/results/phase4_genai_enhancement/`

In [ ]:
P4 = PHASE_DIRS['phase4_genai_enhancement']

# 4.1 CLAHE baseline
_clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
def enhance_clahe(img):
    arr = np.array(img.convert('RGB'))
    lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB)
    lab[..., 0] = _clahe.apply(lab[..., 0])
    return Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB))

In [ ]:
# 4.2 GenAI restoration — Swin2SR primary, TinyU-Net fallback
def _try_swinir():
    try:
        from transformers import Swin2SRForImageSuperResolution, Swin2SRImageProcessor
        proc  = Swin2SRImageProcessor.from_pretrained('caidas/swin2SR-classical-sr-x2-64')
        mdl   = Swin2SRForImageSuperResolution.from_pretrained('caidas/swin2SR-classical-sr-x2-64').to(DEVICE).eval()
        @torch.no_grad()
        def process(img):
            inputs = proc(img, return_tensors='pt').to(DEVICE)
            out = mdl(**inputs).reconstruction
            arr = out.squeeze().clamp(0, 1).cpu().permute(1, 2, 0).numpy()
            return Image.fromarray((arr * 255).astype(np.uint8)).resize(img.size)
        return process
    except Exception as e:
        print('[swinir failed]', e); return None

class TinyUNet(nn.Module):
    def __init__(self, ch=32):
        super().__init__()
        def conv(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.ReLU(True),
                                             nn.Conv2d(o, o, 3, padding=1), nn.ReLU(True))
        self.e1 = conv(3, ch); self.e2 = conv(ch, ch*2); self.e3 = conv(ch*2, ch*4)
        self.up2 = nn.ConvTranspose2d(ch*4, ch*2, 2, stride=2); self.d2 = conv(ch*4, ch*2)
        self.up1 = nn.ConvTranspose2d(ch*2, ch, 2, stride=2);   self.d1 = conv(ch*2, ch)
        self.out = nn.Conv2d(ch, 3, 1)
    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(F.max_pool2d(e1, 2)); e3 = self.e3(F.max_pool2d(e2, 2))
        d2 = self.d2(torch.cat([self.up2(e3), e2], 1))
        d1 = self.d1(torch.cat([self.up1(d2), e1], 1))
        return torch.sigmoid(self.out(d1))

def _train_tiny_unet(epochs=2, n=400):
    df = pd.read_csv(PRISTINE_CSV).sample(n, random_state=0)
    tfm = T.Compose([T.Resize((IMAGE_SIZE, IMAGE_SIZE)), T.ToTensor()])
    mdl = TinyUNet().to(DEVICE)
    optim = torch.optim.AdamW(mdl.parameters(), lr=1e-3); crit = nn.L1Loss()
    for ep in range(epochs):
        for row in df.sample(frac=1).itertuples():
            img = Image.open(APTOS_IMAGES / f'{row.id_code}.png').convert('RGB')
            target = tfm(img).unsqueeze(0).to(DEVICE)
            if np.random.rand() < 0.5:
                noisy = (target + 0.1 * torch.randn_like(target)).clamp(0, 1)
            else:
                k = 5; w = torch.ones((3, 1, k, k), device=DEVICE) / (k*k)
                noisy = F.conv2d(target, w, padding=k//2, groups=3)
            optim.zero_grad(); crit(mdl(noisy), target).backward(); optim.step()
    return mdl

def _make_unet_processor(mdl):
    tfm = T.Compose([T.Resize((IMAGE_SIZE, IMAGE_SIZE)), T.ToTensor()])
    @torch.no_grad()
    def process(img):
        x = tfm(img).unsqueeze(0).to(DEVICE)
        out = mdl(x).squeeze().clamp(0, 1).cpu().permute(1, 2, 0).numpy()
        return Image.fromarray((out * 255).astype(np.uint8)).resize(img.size)
    return process

enhance_genai = _try_swinir()
if enhance_genai is None:
    print('Falling back to TinyU-Net (training ~1 min)...')
    ckpt = CHECKPOINTS_DIR / 'genai_unet.pt'
    unet = TinyUNet().to(DEVICE)
    if ckpt.exists():
        unet.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    else:
        unet = _train_tiny_unet(); torch.save(unet.state_dict(), ckpt)
    enhance_genai = _make_unet_processor(unet)
    GENAI_BACKEND = 'unet'
else:
    GENAI_BACKEND = 'swin2sr'
print('GenAI backend:', GENAI_BACKEND)

In [ ]:
# 4.3 Build enhanced versions of every degraded image (test ids only)
test_id_set = set(pd.read_csv(P2 / 'metrics' / 'test_ids.csv')['id_code'].astype(str))

def build_enhanced(method_name, fn):
    for k in DEGRADATION_TYPES:
        for l in DEGRADATION_LEVELS:
            src_dir = DEGRADED_DIR / k / l
            out_dir = ENHANCED_DIR / method_name / k / l
            out_dir.mkdir(parents=True, exist_ok=True)
            mani = pd.read_csv(src_dir / 'manifest.csv')
            mani = mani[mani['id_code'].astype(str).isin(test_id_set)]
            for _, row in tqdm(mani.iterrows(), total=len(mani),
                                desc=f'{method_name}/{k}/{l}', leave=False):
                out = out_dir / row['rel_path']
                if not out.exists():
                    fn(Image.open(src_dir / row['rel_path']).convert('RGB')).save(out)
            mani.assign(method=method_name).to_csv(out_dir / 'manifest.csv', index=False)

build_enhanced('clahe', enhance_clahe)
build_enhanced('genai', enhance_genai)
print('Enhanced sets ready.')

In [ ]:
# 4.4 Re-evaluate the three classifiers on raw degraded vs CLAHE vs GenAI
ENHANCERS = ('clahe', 'genai')
rows = []
for name in MODEL_NAMES:
    print(f'\n=== {name} ===')
    model = load_classifier(name)
    for k in DEGRADATION_TYPES:
        for l in DEGRADATION_LEVELS:
            for variant in ('raw', *ENHANCERS):
                root = DEGRADED_DIR / k / l if variant == 'raw' else ENHANCED_DIR / variant / k / l
                ds = FolderDataset(root)
                ds.df = ds.df[ds.df['id_code'].astype(str).isin(test_id_set)].reset_index(drop=True)
                m = evaluate(model, DataLoader(ds, batch_size=32, num_workers=2, pin_memory=True))
                rows.append({'model': name, 'degradation': k, 'level': l, 'variant': variant, **m})
                print(f'  {k}/{l}/{variant}: acc={m["accuracy"]:.4f}')
    del model; torch.cuda.empty_cache()

rec_df = pd.DataFrame(rows)
rec_df.to_csv(P4 / 'metrics' / 'recovery_accuracy.csv', index=False)
rec_df.head()

In [ ]:
# 4.5 Recovery plots — accuracy
level_order_no_none = ['low', 'mid', 'high']
for kind in DEGRADATION_TYPES:
    fig, axes = plt.subplots(1, len(MODEL_NAMES), figsize=(4.2*len(MODEL_NAMES), 4), sharey=True)
    for ax, name in zip(axes, MODEL_NAMES):
        sub = rec_df[(rec_df['degradation'] == kind) & (rec_df['model'] == name)].copy()
        sub['level'] = pd.Categorical(sub['level'], categories=level_order_no_none, ordered=True)
        for variant in ('raw', 'clahe', 'genai'):
            d = sub[sub['variant'] == variant].sort_values('level')
            ax.plot(d['level'], d['accuracy'], marker='o', label=variant)
        ax.set_title(name); ax.set_xlabel('level'); ax.set_ylim(0, 1); ax.grid(alpha=0.3)
    axes[0].set_ylabel('accuracy'); axes[-1].legend()
    fig.suptitle(f'Accuracy recovery — {kind}')
    plt.tight_layout()
    out = P4 / 'plots' / f'recovery_accuracy_{kind}.png'
    plt.savefig(out, dpi=150); plt.show(); print('Saved:', out)

In [ ]:
# 4.6 Slim XAI re-evaluation on enhanced images (cheapest method per model)
METHOD_FOR = {'resnet50': ('gradcam', gradcam_heatmap),
               'efficientnet_b3': ('gradcam', gradcam_heatmap),
               'vit_base': ('attention', attention_rollout)}
EXPLAIN_IDS_P4 = list(test_id_set)[:15]

rows = []
for name in MODEL_NAMES:
    method_name, fn = METHOD_FOR[name]
    model = load_classifier(name)
    print(f'\n=== {name} / {method_name} ===')
    for id_code in tqdm(EXPLAIN_IDS_P4):
        if id_code not in labels_df.index: continue
        label = int(labels_df[id_code])
        x_c = load_tensor(APTOS_IMAGES / f'{id_code}.png')
        try: h_c = fn(model, x_c, label)
        except Exception: continue
        for k in DEGRADATION_TYPES:
            for l in DEGRADATION_LEVELS:
                for variant in ('raw', *ENHANCERS):
                    src = (DEGRADED_DIR if variant == 'raw' else ENHANCED_DIR / variant) / k / l / f'{id_code}.png'
                    if not src.exists(): continue
                    x = load_tensor(src)
                    try: h = fn(model, x, label)
                    except Exception: continue
                    rows.append({'id_code': id_code, 'model': name, 'method': method_name,
                                 'degradation': k, 'level': l, 'variant': variant,
                                 'stability': stability_spearman(h_c, h),
                                 'insertion_auc': insertion_auc(model, x, h, label)})
    del model; torch.cuda.empty_cache()

xai_rec = pd.DataFrame(rows)
xai_rec.to_csv(P4 / 'metrics' / 'recovery_xai.csv', index=False)
xai_rec.head()

In [ ]:
# 4.7 Recovery plots — XAI
for metric in ('stability', 'insertion_auc'):
    for kind in DEGRADATION_TYPES:
        sub = xai_rec[xai_rec['degradation'] == kind].copy()
        if sub.empty: continue
        sub['level'] = pd.Categorical(sub['level'], categories=level_order_no_none, ordered=True)
        g = sub.groupby(['model', 'variant', 'level'])[metric].mean().reset_index()
        fig, axes = plt.subplots(1, len(MODEL_NAMES), figsize=(4.2*len(MODEL_NAMES), 4), sharey=True)
        for ax, name in zip(axes, MODEL_NAMES):
            for variant in ('raw', 'clahe', 'genai'):
                d = g[(g['model'] == name) & (g['variant'] == variant)].sort_values('level')
                ax.plot(d['level'], d[metric], marker='o', label=variant)
            ax.set_title(name); ax.grid(alpha=0.3); ax.set_xlabel('level')
        axes[0].set_ylabel(metric); axes[-1].legend()
        fig.suptitle(f'XAI {metric} recovery — {kind}'); plt.tight_layout()
        out = P4 / 'plots' / f'recovery_xai_{metric}_{kind}.png'
        plt.savefig(out, dpi=150); plt.show()

# 4.8 Side-by-side qualitative
demo_id = EXPLAIN_IDS_P4[0]
fig, axes = plt.subplots(len(DEGRADATION_TYPES), 4, figsize=(13, 3.2*len(DEGRADATION_TYPES)))
for r, kind in enumerate(DEGRADATION_TYPES):
    clean = Image.open(APTOS_IMAGES / f'{demo_id}.png').convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
    deg   = Image.open(DEGRADED_DIR / kind / 'high' / f'{demo_id}.png').convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
    cl    = Image.open(ENHANCED_DIR / 'clahe' / kind / 'high' / f'{demo_id}.png').convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
    gn    = Image.open(ENHANCED_DIR / 'genai' / kind / 'high' / f'{demo_id}.png').convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
    for ax, im, ttl in zip(axes[r], (clean, deg, cl, gn), ('clean', f'{kind}-high', 'CLAHE', 'GenAI')):
        ax.imshow(im); ax.set_title(ttl); ax.axis('off')
plt.tight_layout()
out = P4 / 'samples' / f'recovery_{demo_id}.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print('Saved:', out)

---
# Phase 5 — Quality-Aware Ensemble Framework (RQ4)
Outputs: `Drive/Thesis/results/phase5_quality_ensemble/`

In [ ]:
P5 = PHASE_DIRS['phase5_quality_ensemble']

# 5.1 Train (or load) an EyeQ quality classifier (good/usable/reject)
class EyeQDataset(Dataset):
    def __init__(self, csv_path, images_root, transform,
                 id_col='image', quality_col='quality'):
        self.df = pd.read_csv(csv_path); self.df[id_col] = self.df[id_col].astype(str)
        self.images_root = Path(images_root); self.transform = transform
        self.id_col, self.quality_col = id_col, quality_col
        all_imgs = {p.stem: p for p in self.images_root.rglob('*.*')
                    if p.suffix.lower() in ('.png', '.jpg', '.jpeg')}
        self.df['__path'] = self.df[id_col].apply(lambda s: all_imgs.get(Path(s).stem))
        self.df = self.df.dropna(subset=['__path']).reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row['__path']).convert('RGB')
        return self.transform(img), int(row[self.quality_col])

def build_quality_model(): return timm.create_model('resnet18', pretrained=True, num_classes=3)
Q_CKPT = CHECKPOINTS_DIR / 'quality_classifier.pt'

def train_quality(epochs=2, bs=32):
    if EYEQ_CSV is None: return None
    ds = EyeQDataset(EYEQ_CSV, EYEQ_DIR, TFM_TRAIN)
    n_val = max(1, int(0.15 * len(ds)))
    tds, vds = random_split(ds, [len(ds)-n_val, n_val], generator=torch.Generator().manual_seed(0))
    vds.dataset = EyeQDataset(EYEQ_CSV, EYEQ_DIR, TFM_EVAL)
    tl = DataLoader(tds, batch_size=bs, shuffle=True,  num_workers=2)
    vl = DataLoader(vds, batch_size=bs, shuffle=False, num_workers=2)
    mdl = build_quality_model().to(DEVICE)
    optim = torch.optim.AdamW(mdl.parameters(), lr=3e-4, weight_decay=1e-4); crit = nn.CrossEntropyLoss()
    for ep in range(epochs):
        mdl.train()
        for x, y in tl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optim.zero_grad(); crit(mdl(x), y).backward(); optim.step()
        mdl.eval(); ys, ps = [], []
        with torch.no_grad():
            for x, y in vl:
                ps.append(mdl(x.to(DEVICE)).argmax(1).cpu().numpy()); ys.append(y.numpy())
        y = np.concatenate(ys); p = np.concatenate(ps)
        print(f'epoch {ep+1}: val acc = {accuracy_score(y, p):.4f}')
    rep = classification_report(y, p, target_names=list(EYEQ_QUALITY_LABELS.values()),
                                 output_dict=True, zero_division=0)
    torch.save(mdl.state_dict(), Q_CKPT)
    with open(P5 / 'metrics' / 'quality_classifier_metrics.json', 'w') as f:
        json.dump(rep, f, indent=2)
    return mdl

if Q_CKPT.exists():
    qmodel = build_quality_model().to(DEVICE)
    qmodel.load_state_dict(torch.load(Q_CKPT, map_location=DEVICE)); qmodel.eval()
    print('Loaded existing quality classifier.')
else:
    qmodel = train_quality()

In [ ]:
# 5.2 Pick best_clean and best_robust from Phase 2 results
stress_df = pd.read_csv(P2 / 'metrics' / 'stress_test_results.csv')
best_clean  = stress_df[stress_df['degradation'] == 'clean'].sort_values('accuracy', ascending=False).iloc[0]['model']
best_robust = stress_df[stress_df['degradation'] != 'clean'].groupby('model')['accuracy'].mean().idxmax()
print('best_clean :', best_clean)
print('best_robust:', best_robust)

ROUTES = {
    'good':   {'enhancement': 'none',  'model': best_clean,  'xai': 'attention' if best_clean  == 'vit_base' else 'gradcam'},
    'usable': {'enhancement': 'clahe', 'model': best_clean,  'xai': 'attention' if best_clean  == 'vit_base' else 'gradcam'},
    'reject': {'enhancement': 'genai', 'model': best_robust, 'xai': 'attention' if best_robust == 'vit_base' else 'gradcam'},
}
ROUTES

In [ ]:
# 5.3 Pipeline function (quality -> route -> enhance -> classify -> XAI -> trust)
classifiers = {n: load_classifier(n) for n in {best_clean, best_robust}}
USE_PRECOMPUTED_GENAI = (ENHANCED_DIR / 'genai').exists()

def predict_quality(img):
    if qmodel is None: return 'good'
    x = TFM_EVAL(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        return EYEQ_QUALITY_LABELS[int(qmodel(x).argmax(1).item())]

def apply_enhancement(img, kind, kind_hint=None, level_hint=None, id_hint=None):
    if kind == 'none':  return img
    if kind == 'clahe': return enhance_clahe(img)
    if kind == 'genai':
        if USE_PRECOMPUTED_GENAI and kind_hint and level_hint and id_hint:
            cached = ENHANCED_DIR / 'genai' / kind_hint / level_hint / f'{id_hint}.png'
            if cached.exists(): return Image.open(cached).convert('RGB')
        return enhance_genai(img)
    return img

XAI_FNS = {'gradcam': gradcam_heatmap, 'attention': attention_rollout}

def trust_score(conf, insertion): return float(0.5 * conf + 0.5 * insertion)

def pipeline(img, kind_hint=None, level_hint=None, id_hint=None):
    quality  = predict_quality(img)
    route    = ROUTES[quality]
    enhanced = apply_enhancement(img, route['enhancement'],
                                  kind_hint=kind_hint, level_hint=level_hint, id_hint=id_hint)
    model = classifiers[route['model']]
    x = TFM_EVAL(enhanced).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        prob = torch.softmax(model(x), 1)[0]
        pred = int(prob.argmax().item()); conf = float(prob[pred].item())
    heat = XAI_FNS[route['xai']](model, x, target_class=pred)
    ins  = insertion_auc(model, x, heat, pred)
    return {'quality_pred': quality, 'route': route, 'pred': pred, 'conf': conf,
            'insertion_auc': ins, 'trust': trust_score(conf, ins),
            'heatmap': heat, 'enhanced_img': enhanced}

In [ ]:
# 5.4 Run pipeline on test set: clean + blur-high + exposure-high
EVAL_CONDITIONS = [('clean', None, None),
                    ('blur',     'blur',     'high'),
                    ('exposure', 'exposure', 'high')]

rows = []
for id_code in tqdm(list(test_id_set)[:80], desc='ensemble'):    # cap 80 for time
    if id_code not in labels_df.index: continue
    label = int(labels_df[id_code])
    for tag, kind, level in EVAL_CONDITIONS:
        src = APTOS_IMAGES / f'{id_code}.png' if tag == 'clean' \
              else DEGRADED_DIR / kind / level / f'{id_code}.png'
        if not src.exists(): continue
        out = pipeline(Image.open(src).convert('RGB'),
                       kind_hint=kind, level_hint=level, id_hint=id_code)
        rows.append({'id_code': id_code, 'condition': tag, 'true_label': label,
                     'pred': out['pred'], 'correct': int(out['pred'] == label),
                     'quality_pred': out['quality_pred'],
                     'route_model': out['route']['model'],
                     'route_enhancement': out['route']['enhancement'],
                     'route_xai': out['route']['xai'],
                     'conf': out['conf'], 'insertion_auc': out['insertion_auc'],
                     'trust': out['trust']})

ens_df = pd.DataFrame(rows)
ens_df.to_csv(P5 / 'metrics' / 'ensemble_predictions.csv', index=False)
ens_df.head()

In [ ]:
# 5.5 Aggregate vs single-best baseline (Phase 2)
summary = (ens_df.groupby('condition')
                  .agg(accuracy=('correct', 'mean'),
                       mean_trust=('trust', 'mean'),
                       mean_insertion=('insertion_auc', 'mean'),
                       n=('id_code', 'count'))).round(4).reset_index()
baseline = (stress_df[stress_df['model'] == best_clean]
             .assign(condition=lambda d: np.where(d['degradation'] == 'clean', 'clean',
                                                  d['degradation'] + '-' + d['level']))
             [['condition', 'accuracy', 'f1_macro']]
             .rename(columns={'accuracy': 'baseline_acc', 'f1_macro': 'baseline_f1'}))
summary = summary.merge(baseline, on='condition', how='left')
summary.to_csv(P5 / 'metrics' / 'ensemble_summary.csv', index=False)
summary

In [ ]:
# 5.6 Plots
fig, ax = plt.subplots(figsize=(7, 4))
for cond in ens_df['condition'].unique():
    ax.hist(ens_df.loc[ens_df['condition'] == cond, 'trust'], bins=15, alpha=0.5, label=cond)
ax.set_title('Clinical Trust Score distribution'); ax.set_xlabel('trust'); ax.legend()
plt.tight_layout()
plt.savefig(P5 / 'plots' / 'trust_score_distribution.png', dpi=150); plt.show()

clean = ens_df[ens_df['condition'] == 'clean']
if not clean.empty:
    cm = confusion_matrix(clean['true_label'], clean['pred'], labels=list(range(NUM_CLASSES)))
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap='Blues')
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > cm.max()/2 else 'black')
    ax.set_xlabel('predicted'); ax.set_ylabel('true')
    ax.set_title('Confusion matrix — ensemble (clean)'); fig.colorbar(im)
    plt.tight_layout()
    plt.savefig(P5 / 'plots' / 'confusion_matrix.png', dpi=150); plt.show()

In [ ]:
# 5.7 Qualitative — one routed example
demo = ens_df.iloc[0]
src = APTOS_IMAGES / f"{demo['id_code']}.png" if demo['condition'] == 'clean' \
      else DEGRADED_DIR / demo['condition'] / 'high' / f"{demo['id_code']}.png"
img = Image.open(src).convert('RGB')
out = pipeline(img,
               kind_hint=None if demo['condition'] == 'clean' else demo['condition'],
               level_hint=None if demo['condition'] == 'clean' else 'high',
               id_hint=demo['id_code'])
fig, axes = plt.subplots(1, 3, figsize=(11, 4))
axes[0].imshow(img.resize((IMAGE_SIZE, IMAGE_SIZE))); axes[0].set_title(f'input ({demo["condition"]})')
axes[1].imshow(out['enhanced_img'].resize((IMAGE_SIZE, IMAGE_SIZE)))
axes[1].set_title(f"after {out['route']['enhancement']}")
axes[2].imshow(out['enhanced_img'].resize((IMAGE_SIZE, IMAGE_SIZE)))
axes[2].imshow(out['heatmap'], cmap='jet', alpha=0.45)
axes[2].set_title(f"{out['route']['xai']} — pred={out['pred']} true={demo['true_label']}")
for ax in axes: ax.axis('off')
fig.suptitle(f"Q={out['quality_pred']} | M={out['route']['model']} | trust={out['trust']:.2f}")
plt.tight_layout()
save = P5 / 'samples' / f"route_{demo['id_code']}_{demo['condition']}.png"
plt.savefig(save, dpi=150, bbox_inches='tight'); plt.show()
print('Saved:', save)

---
## All five phases complete

Drive layout after a full run:
```
Thesis/
├── Thesis.zip
├── checkpoints/                          # *.pt for ResNet50, EfficientNet-B3, ViT-Base, quality classifier
└── results/
    ├── phase1_data_engineering/{metrics,plots,samples,logs}
    ├── phase2_model_benchmarking/...
    ├── phase3_xai_benchmark/...
    ├── phase4_genai_enhancement/...
    └── phase5_quality_ensemble/...
```
Each phase folder is fully self-contained — share any one with the professor without dragging in the others.